# NBA Playoff predictor
- We will train our model based off of 20 years of season data
- Features we will select include otrg, dtrg, pace, rest_days
- TRAIN_SEASONS = 2003-04 to 2018-19 (Large Sample good for model training)
- TEST_SEASONS  = 2019-20 to 2023-24 (Recent seasons would be better for not generalising)

## Updates:
### 1/06/2026 
- We're gonna try to include first rounds. Even though in first round series, upsets are quite rare among matchups, but it expands the dataset, and gives us more information showing that higher seeded teams are more likely to win their series.
    - This gives us a total of 315 series matchups to work with starting from 2003-04 to 2023-24 (20 seasons)
    - From matchups in the first round to the NBA finals
- Remove NET_RTG from featue selection, as we already have ORTG and DRTG (multicolinearity reasons)

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from nba_api.stats.endpoints import leaguegamefinder, leaguestandings, leaguedashteamstats
import time
from requests.exceptions import Timeout, ConnectionError


SEASONS = [f'{y}-{str(y+1)[-2:]}' for y in range(2014, 2024)]
DELAY = 5
OUT_DIR = Path('data')
OUT_DIR.mkdir(exist_ok=True)

HEADERS = {
    'Host':                  'stats.nba.com',
    'User-Agent':            'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Accept':                'application/json, text/plain, */*',
    'Accept-Language':       'en-US,en;q=0.9',
    'Referer':               'https://www.nba.com/',
    'x-nba-stats-origin':   'stats',
    'x-nba-stats-token':    'true',
    'Connection':            'keep-alive',

}

def scrape_playoff_games(seasons):
    all_games = []

    for season in seasons:
        try:
            resp = leaguegamefinder.LeagueGameFinder(
                season_nullable= season,
                season_type_nullable='Playoffs',
                league_id_nullable='00',
                player_or_team_abbreviation='T',
                headers=HEADERS,
                timeout=30   
            ).get_data_frames()[0]
        
            if resp.empty:
                print(f'[WARN]: No data for {season}')
                continue

            resp['SEASON'] = season
            all_games.append(
                resp[['SEASON', 'TEAM_ID', 'TEAM_ABBREVIATION',
                      'GAME_ID','GAME_DATE','MATCHUP','WL'
                      ]]
            )
        except Exception as e:
            print(f'[ERROR] {season}: {e}')
        
        time.sleep(DELAY)

    df = pd.concat(all_games,ignore_index=True)
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    df = df.sort_values(['SEASON', 'GAME_DATE', 'GAME_ID']).reset_index(drop=True)

    path = OUT_DIR / 'playoff_games.csv'
    df.to_csv(path, index=False)
    df.head(10)
    return df



In [2]:
def scrape_seedings(seasons):
    all_standings = []

    for season in seasons:
        try:
            resp = leaguestandings.LeagueStandings(
                season = season,
                season_type='Regular Season',
                league_id='00',
                headers=HEADERS,
                timeout=30
            ).get_data_frames()[0]
            SEED_CANDIDATES = ['PlayoffSeeding', 'PlayoffRank', 'PLAYOFF_SEED']
            seed_col = next(
                (c for c in SEED_CANDIDATES if c in resp.columns),
                None
            )
            if seed_col is None:
                print(f'[WARN] no seed column found for {season}, cols: {resp.columns.tolist()}')
                continue

            resp = resp[['TeamID', seed_col]].copy()
            resp.columns = ['TEAM_ID', 'PLAYOFF_SEED']
            resp['SEASON'] = season
            all_standings.append(resp)
            
        except Exception as e:
            print(f'[ERROR] {season}: {e}')

        time.sleep(DELAY)

    df = pd.concat(all_standings, ignore_index=True)
    df['PLAYOFF_SEED'] = pd.to_numeric(df['PLAYOFF_SEED'], errors='coerce')
    df.head(10)
    return df

In [3]:
def reconstruct_series(games, seedings):
    games = games.copy()
    games['GAME_DATE']  = pd.to_datetime(games['GAME_DATE'])
    games['GAME_ID']    = games['GAME_ID'].astype(str)
    games['ROUND']      = games['GAME_ID'].astype(str).str[5].astype(int)
    games['SERIES_KEY'] = games['GAME_ID'].astype(str).str[:9]

    def last_game_before(team_id, season, before_date):
        prior = games[
            (games['TEAM_ID']   == team_id) &
            (games['SEASON']    == season)  &
            (games['GAME_DATE'] <  before_date)
        ]['GAME_DATE']
        return prior.max() if not prior.empty else None

    series_rows = []
    skipped     = 0

    for (season, series_key), grp in games.groupby(['SEASON', 'SERIES_KEY']):
        team_ids = grp['TEAM_ID'].unique()
        if len(team_ids) != 2:
            continue

        team_a_id, team_b_id = team_ids
        wins_a = len(grp[(grp['TEAM_ID'] == team_a_id) & (grp['WL'] == 'W')])
        wins_b = len(grp[(grp['TEAM_ID'] == team_b_id) & (grp['WL'] == 'W')])
        if wins_a != 4 and wins_b != 4:
            continue

        winner_id    = team_a_id if wins_a == 4 else team_b_id
        games_played = wins_a + wins_b
        series_start = grp['GAME_DATE'].min()
        round_num    = grp['ROUND'].iloc[0]

        # ── Seeding ────────────────────────────────────────────────────
        seed_lookup = (
            seedings[seedings['SEASON'] == season]
            .set_index('TEAM_ID')['PLAYOFF_SEED']
            .to_dict()
        )
        seed_a = seed_lookup.get(team_a_id)
        seed_b = seed_lookup.get(team_b_id)

        if seed_a is None or seed_b is None:
            skipped += 1
            print(f"  [SKIP] {season} series {series_key} (round {round_num}) — missing seed for "
                  f"{'team A' if seed_a is None else 'team B'}")
            continue

        # Ensure A = higher seed (lower number)
        if seed_a > seed_b:
            team_a_id, team_b_id = team_b_id, team_a_id
            seed_a,    seed_b    = seed_b,    seed_a

        higher_seed_wins = int(winner_id == team_a_id)

        # ── Rest days ──────────────────────────────────────────────────
        last_a = last_game_before(team_a_id, season, series_start)
        last_b = last_game_before(team_b_id, season, series_start)
        rest_a = (series_start - last_a).days if last_a is not None else 7
        rest_b = (series_start - last_b).days if last_b is not None else 7

        # ── Abbreviations ──────────────────────────────────────────────
        abbrev = grp.set_index('TEAM_ID')['TEAM_ABBREVIATION'].to_dict()

        series_rows.append({
            'SEASON':           season,
            'SERIES_KEY':       series_key,
            'ROUND':            round_num,
            'SERIES_START':     series_start,
            'TEAM_A_ID':        team_a_id,
            'TEAM_B_ID':        team_b_id,
            'TEAM_A_ABB':       abbrev.get(team_a_id),
            'TEAM_B_ABB':       abbrev.get(team_b_id),
            'SEED_A':           seed_a,
            'SEED_B':           seed_b,
            'WINS_A':           wins_a if winner_id == team_a_id else wins_b,
            'WINS_B':           wins_b if winner_id == team_a_id else wins_a,
            'GAMES_PLAYED':     games_played,
            'WINNER_ID':        winner_id,
            'HIGHER_SEED_WINS': higher_seed_wins,
            'REST_A':           rest_a,
            'REST_B':           rest_b,
            'REST_DIFF':        rest_a - rest_b,
        })

    df = pd.DataFrame(series_rows).sort_values(['SEASON', 'SERIES_START'])
    path = OUT_DIR / 'playoff_series.csv'
    df.to_csv(path, index=False)
    print(f"  Saved {len(df)} series ({skipped} skipped) → {path}")
    print(f"  Round breakdown:\n{df.groupby('ROUND').size().rename({1:'R1',2:'R2',3:'Conf Finals',4:'Finals'})}")
    return df

In [4]:
def scrape_reg_seasons_stats(seasons, max_retries=3, timeout=60):
    all_stats = []

    for season in seasons:
        for attempt in range(1, max_retries + 1):
            try:
                df = leaguedashteamstats.LeagueDashTeamStats(
                    season=season,
                    season_type_all_star='Regular Season',
                    measure_type_detailed_defense='Advanced',
                    per_mode_detailed='PerGame',
                    headers=HEADERS,
                    timeout=timeout
                ).get_data_frames()[0]

                col_map = {}
                for c in df.columns:
                    cu = c.upper()
                    if 'OFF_RATING' in cu and 'ORTG'    not in col_map.values():
                        col_map[c] = 'ORTG'
                    elif 'DEF_RATING' in cu and 'DRTG'  not in col_map.values():
                        col_map[c] = 'DRTG'
                    elif cu in ('PACE', 'E_PACE') and 'PACE' not in col_map.values():
                        col_map[c] = 'PACE'
                    elif 'NET_RATING' in cu and 'NET_RTG' not in col_map.values():
                        col_map[c] = 'NET_RTG'

                df = df.rename(columns=col_map)

                required = {'ORTG', 'DRTG', 'PACE'}
                missing  = required - set(df.columns)
                if missing:
                    print(f'  [WARN] {season}: missing {missing} — skipping')
                    break

                df['SEASON'] = season
                keep = [c for c in
                        ['SEASON', 'TEAM_ID', 'TEAM_NAME', 'W', 'L', 'W_PCT',
                         'ORTG', 'DRTG', 'PACE', 'NET_RTG']
                        if c in df.columns]
                all_stats.append(df[keep])
                print(f'  [OK] {season}')
                break

            except (Timeout, ConnectionError):
                wait = attempt * 10
                print(f'  [TIMEOUT] {season} attempt {attempt}/{max_retries} — retrying in {wait}s')
                time.sleep(wait)

            except Exception as e:
                print(f'  [ERROR] {season}: {e}')
                break
        else:
            print(f'  [FAILED] {season}: all {max_retries} attempts timed out — skipping')

        time.sleep(DELAY)

    stats = pd.concat(all_stats, ignore_index=True)
    stats = stats.drop_duplicates(subset=['SEASON', 'TEAM_ID'], keep='first')
    path  = OUT_DIR / 'reg_season_stats.csv'
    stats.to_csv(path, index=False)
    print(f'  Saved {len(stats)} rows → {path}')
    return stats

In [5]:

def build_series_features(series_path, stats):
    series = pd.read_csv(series_path, parse_dates=["SERIES_START"])

    dupes = stats.duplicated(subset=["SEASON", "TEAM_ID"]).sum()
    if dupes:
        print(f"  [WARN] {dupes} duplicate (SEASON, TEAM_ID) pairs — deduplicating, keeping last")

    stats_idx = (
        stats
        .assign(
            SEASON  = stats["SEASON"].astype(str),
            TEAM_ID = stats["TEAM_ID"].astype(int),
        )
        .drop_duplicates(subset=["SEASON", "TEAM_ID"], keep="last")
        .set_index(["SEASON", "TEAM_ID"])
    )

    has_rest = "REST_DIFF" in series.columns
    rows     = []
    skipped  = 0

    for _, s in series.iterrows():
        season = str(s["SEASON"])
        a_id   = int(s["TEAM_A_ID"])
        b_id   = int(s["TEAM_B_ID"])

        try:
            a = stats_idx.loc[(season, a_id)]
            b = stats_idx.loc[(season, b_id)]
        except KeyError:
            skipped += 1
            continue

        has_net = "NET_RTG" in a.index

        rows.append({
            "SEASON":           season,
            "SERIES_KEY":       s["SERIES_KEY"],
            "SERIES_START":     s["SERIES_START"],
            "TEAM_A_ABB":       s["TEAM_A_ABB"],
            "TEAM_B_ABB":       s["TEAM_B_ABB"],
            "SEED_A":           s["SEED_A"],
            "SEED_B":           s["SEED_B"],
            "ORTG_A":           float(a["ORTG"]),
            "DRTG_A":           float(a["DRTG"]),
            "PACE_A":           float(a["PACE"]),
            "W_PCT_A":          float(a["W_PCT"]),
            "ORTG_B":           float(b["ORTG"]),
            "DRTG_B":           float(b["DRTG"]),
            "PACE_B":           float(b["PACE"]),
            "W_PCT_B":          float(b["W_PCT"]),
            "ORTG_DIFF":        float(a["ORTG"])    - float(b["ORTG"]),
            "DRTG_DIFF":        float(a["DRTG"])    - float(b["DRTG"]),
            "PACE_DIFF":        float(a["PACE"])    - float(b["PACE"]),
            "NET_RTG_DIFF":     float(a["NET_RTG"]) - float(b["NET_RTG"]) if has_net else np.nan,
            "W_PCT_DIFF":       float(a["W_PCT"])   - float(b["W_PCT"]),
            "SEED_DIFF":        int(s["SEED_A"])    - int(s["SEED_B"]),
            "REST_DIFF":        float(s["REST_DIFF"]) if has_rest else np.nan,
            "HIGHER_SEED_WINS": s["HIGHER_SEED_WINS"],
        })

    features = pd.DataFrame(rows).sort_values(["SEASON", "SERIES_START"])
    features.to_csv(OUT_DIR / "series_features.csv", index=False)
    print(f"  Saved {len(features)} series → {OUT_DIR / 'series_features.csv'}")
    if skipped:
        print(f"  Skipped {skipped} series (missing stats)")
    return features

In [6]:
def validate_features(df: pd.DataFrame) -> None:
    feature_cols = ['ORTG_DIFF', 'DRTG_DIFF', 'PACE_DIFF', 'NET_RTG_DIFF',
                    'W_PCT_DIFF', 'SEED_DIFF']
 
    print(f'\n  Shape:  {df.shape}')
    print(f'  Seasons: {df['SEASON'].nunique()}  ({df['SEASON'].min()} → {df['SEASON'].max()})')
 
    print(f'\n  Target distribution:')
    vc = df['HIGHER_SEED_WINS'].value_counts(normalize=True)
    print(f'    Higher seed wins: {vc.get(1, 0):.1%}')
    print(f'    Upset:            {vc.get(0, 0):.1%}')
 
    print(f'\n  Nulls per feature:')
    print(df[feature_cols].isnull().sum().to_string())
 
    print(f'\n  Feature summary stats:')
    print(df[feature_cols].describe().round(2).to_string())
 
    bad_seeds = (df['SEED_DIFF'] > 0).sum()
    if bad_seeds:
        print(f'\n  [WARN]: {bad_seeds} rows where SEED_DIFF > 0 (team orientation wrong)')
    else:
        print(f'\n  SEED_DIFF always <= 0 (team orientation correct)')



In [7]:


games = scrape_playoff_games(SEASONS)
seedings = scrape_seedings(SEASONS)
series = reconstruct_series(games,seedings)
scrape_reg_seasons_stats(SEASONS)

stats = pd.read_csv('./data/reg_season_stats.csv')
build_series_features(OUT_DIR / 'playoff_series.csv', stats)
# features = build_series_features(OUT_DIR / 'playoff_series.csv', stats)
# validate_features(features)

  Saved 150 series (0 skipped) → data/playoff_series.csv
  Round breakdown:
ROUND
0    150
dtype: int64
  [OK] 2014-15
  [OK] 2015-16
  [OK] 2016-17
  [OK] 2017-18
  [OK] 2018-19
  [OK] 2019-20
  [OK] 2020-21
  [OK] 2021-22
  [OK] 2022-23
  [OK] 2023-24
  Saved 300 rows → data/reg_season_stats.csv
  Saved 150 series → data/series_features.csv


,SEASON,SERIES_KEY,SERIES_START,TEAM_A_ABB,TEAM_B_ABB,SEED_A,SEED_B,ORTG_A,DRTG_A,PACE_A,...,PACE_B,W_PCT_B,ORTG_DIFF,DRTG_DIFF,PACE_DIFF,NET_RTG_DIFF,W_PCT_DIFF,SEED_DIFF,REST_DIFF,HIGHER_SEED_WINS
0,2014-15,4140012,2015-04-18,CHI,MIL,3,6,104.7,101.5,95.4,...,96.5,0.500,4.2,2.2,-1.1,2.1,0.110,-3,0.0,1
1,2014-15,4140013,2015-04-18,TOR,WAS,4,5,108.1,104.8,95.4,...,96.0,0.561,6.3,4.8,-0.6,1.3,0.037,-1,0.0,0
2,2014-15,4140014,2015-04-18,GSW,NOP,1,8,109.7,98.2,100.7,...,93.7,0.549,4.3,-6.5,7.0,10.7,0.268,-7,0.0,1
3,2014-15,4140015,2015-04-18,HOU,DAL,2,7,104.2,100.5,99.3,...,97.4,0.610,-3.0,-3.2,1.9,0.2,0.073,-5,0.0,1
4,2014-15,4140010,2015-04-19,ATL,BKN,1,8,106.2,100.7,96.2,...,95.0,0.463,4.3,-4.3,1.2,8.7,0.269,-7,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,2023-24,4230020,2024-05-07,BOS,CLE,1,4,120.2,109.0,99.5,...,99.5,0.585,7.4,-0.9,0.0,8.3,0.195,-3,4.0,1
146,2023-24,4230022,2024-05-07,OKC,DAL,1,5,117.0,109.1,102.3,...,102.4,0.610,1.8,-3.5,-0.1,5.3,0.085,-4,4.0,0
147,2023-24,4230030,2024-05-21,BOS,IND,1,6,120.2,109.0,99.5,...,104.2,0.573,2.3,-6.5,-4.7,8.7,0.207,-5,4.0,1
148,2023-24,4230031,2024-05-22,MIN,DAL,3,5,113.1,106.2,99.5,...,102.4,0.610,-2.1,-6.4,-2.9,4.3,0.073,-2,-1.0,0
